# 02 · Results: experiment table, optimizer study, learning curves
Owner: Experimentation Engineer (spec 3.5). Validation numbers only; the test set stays locked.

**Colab:** Runtime → Change runtime type → **T4 GPU**. Upload the project zip to the Files panel, then Run all.

No data download needed: this notebook only reads `experiments/`.

In [ ]:
# --- Setup: works on Colab and locally. Run this cell first. ---
# It finds the project folder (the one that has src/ and configs/), moves into it,
# and adds it to sys.path so that `from src ... import ...` works.
import os, sys, glob, zipfile

def is_root(path):
    """A folder is the project root if it contains both src/ and configs/."""
    return os.path.isdir(os.path.join(path, "src")) and os.path.isdir(os.path.join(path, "configs"))

def find_root():
    """Look in the current folder, its parent, /content and Google Drive (2 levels deep)."""
    here = os.getcwd()
    candidates = [here, os.path.dirname(here)]
    for base in ["/content", "/content/drive/MyDrive"]:
        candidates += glob.glob(base + "/*") + glob.glob(base + "/*/*")
    return next((c for c in candidates if is_root(c)), None)

IN_COLAB = "google.colab" in sys.modules
root = find_root()

# Colab, option 1: you uploaded the project zip to the Files panel (/content).
if root is None and IN_COLAB:
    for z in glob.glob("/content/*.zip"):
        zipfile.ZipFile(z).extractall("/content/project")
        print("unzipped", z)
    root = find_root()

# Colab, option 2: the project is in your Google Drive.
if root is None and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    root = find_root()

if root is None:
    raise FileNotFoundError("Project folder not found. Upload the project zip (or the repo zip from GitHub: Code -> Download ZIP) to the Colab Files "
                            "panel, or put the project folder in Google Drive, then run this cell again.")
os.chdir(root)
sys.path.insert(0, root)
print("project root:", root)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from src.experiment_log import load_all_runs, plot_curves
runs = load_all_runs()
runs[["run_id", "author", "model", "config", "seed", "params", "epochs_run", "train_time_s",
      "val_balanced_accuracy", "val_macro_f1", "val_recall_mel"]]

## 1. Baselines vs Models A, B, C (validation)

In [ ]:
main = runs[runs["config"].str.contains("baseline|model_")]
main.groupby("config")[["val_balanced_accuracy", "val_recall_mel", "val_macro_f1"]].agg(["mean", "count"]).round(3)

## 2. Optimizer study on Model A (SGD vs SGD + momentum vs Adam)

In [ ]:
opt = runs[runs["config"].isin(["configs/opt_sgd.yaml", "configs/opt_sgd_momentum.yaml",
                                 "configs/model_a_cnn.yaml"])].copy()
opt["optimizer"] = opt["config"].map({"configs/opt_sgd.yaml": "SGD (lr 0.01)",
                                      "configs/opt_sgd_momentum.yaml": "SGD + momentum (lr 0.01)",
                                      "configs/model_a_cnn.yaml": "Adam (lr 0.001)"})
opt[["optimizer", "run_id", "epochs_run", "train_time_s", "val_balanced_accuracy", "val_recall_mel"]]

In [ ]:
latest = opt.groupby("optimizer")["run_id"].last()
plot_curves(list(latest.values), labels=list(latest.index)); plt.show()

**TODO:** which optimizer converges fastest and to the best validation loss? Choose the optimizer from this evidence.

## 3. Learning curves of Models A, B, C

In [ ]:
abc = runs[runs["config"].isin(["configs/model_a_cnn.yaml", "configs/model_b_vgg16_frozen.yaml",
                                 "configs/model_c_vgg16_finetune.yaml"])]
latest = abc.groupby("model")["run_id"].last()
for model, run_id in latest.items():
    plot_curves([run_id], labels=[model]); plt.show()

**TODO:** for each model: overfitting (val loss goes up while train loss goes down), underfitting, or fine? At which epoch did early stopping keep the weights?